## Status 
- twitter business unit and aggregated services is currently calculated by using minnie's dataset (helper/tw_minnie_preBU.csv)
- also minnie's weekly gnl dataset is missing week 51 & 52? 
- 

In [1]:
from IPython.display import display

from tqdm import tqdm 
from datetime import datetime

import pandas as pd
pd.set_option('display.float_format', '{:.00f}'.format)

import numpy as np

import missingno as msno

In [2]:
platformID = 'FBE'

## import helper

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

import test_functions 
from functions import lookup_loader, include_uk_decision, sainsbury_formula, summary_excel, calculate_weekly_Services, calculate_aggregated_services

In [4]:
lookup = lookup_loader(gam_info, platformID, '4',
                       with_country=True, with_pop_col=True)
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']


✅ Test FBE_4_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_4_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_4_02 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_4_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_4_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_4_05 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_4_06 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_4_07 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_4_08 passed: No missing values in lookup.
...updating logbook...



In [5]:
# country
pop_size_col = gam_info['population_column']

channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)

# overlaps 
overlaps = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='overlap')


## import data 

In [6]:
full_df = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_uniqueViewer_country.csv")
                      

full_df.head()

,ServiceID,Channel ID,w/c,PlaceID,uv_by_country
0,DAR,FBE100163990128209,2025-03-31,KUW,378
1,DAR,FBE100163990128209,2025-03-31,BEL,994
2,DAR,FBE100163990128209,2025-03-31,IRQ,907
3,DAR,FBE100163990128209,2025-03-31,CAN,2634
4,DAR,FBE100163990128209,2025-03-31,DEN,386


In [7]:

full_df = full_df.merge(country_codes, on='PlaceID', 
                        how='left', indicator=True)
print(full_df._merge.value_counts())
full_df.drop(columns=['_merge'], inplace=True)


_merge
both          167238
left_only          0
right_only         0
Name: count, dtype: int64


## functions

# calculate 

In [8]:
path = f"../data/singlePlatform/{platformID}/"

## Business Units

In [9]:

data = {}
'''temp_bus = ['GNL', 'WSL', 'GNL','Studios', 'WSE', 'MA-', 'FOA']
for bu in temp_bus:
'''#
for bu in gam_info['business_units'].keys():
    print(f"### processing {bu} ######################################################")
    data[bu] = {'weekly': pd.DataFrame()
               }
    
    bu_configs = gam_info['business_units'][bu]
    print(bu_configs)
    df = full_df[full_df['ServiceID'].isin(gam_info['business_units'][bu]['Service IDs'])]
    
    if df.empty:
        print(f"no data yet for {bu}")
        
    channel_ids = df['Channel ID'].unique().tolist()
    
    # will include / exclude the uk based on bu_configs
    df = include_uk_decision(df, socialmedia_accounts)
    
    # for later testing or if sainsbury isn't used 
    summed_uv_by_country = df.groupby(['ServiceID', 'w/c', 'PlaceID'])\
                                .agg({'uv_by_country': 'sum'})\
                                .reset_index()
    
    if bu_configs['sainsbury'][platformID]:
        print('sainsbury is applied')
        # pivot 
        channel_uv_by_country = pd.crosstab(
                                        index = [ df['PlaceID'], 
                                                  df['ServiceID'], 
                                                  df[pop_size_col], 
                                                  df['w/c']],
                                        columns = df['Channel ID'],
                                        values =  df['uv_by_country'],
                                        aggfunc='sum'
                                    ).reset_index()
    
        # check for missing values
        # especially in the string columns no values should be missing
        #msno.matrix(channel_uv_by_country)
        
        # fill missing values with 0 - this is good fi the matrix above showed that the string 
        # columns did not have any missings so the only gaps filled are numeric. 
        channel_uv_by_country = channel_uv_by_country.fillna(0)
        
        #calculate sainsbury
        channel_uv_by_country = sainsbury_formula(channel_uv_by_country, pop_size_col, 
                                      channel_ids, 'uv_by_country')
        channel_uv_by_country = channel_uv_by_country.drop(columns=channel_ids)
        
        cols_left =  ['w/c', 'PlaceID', 'uv_by_country']
        cols_right = ['w/c', 'PlaceID', 'ServiceID', 'uv_by_country']
        #yt_deduped = channel_uv_by_country[cols_left].merge(summed_uv_by_country[cols_right], on=['w/c', 'PlaceID'], how='inner')
        yt_deduped = channel_uv_by_country.rename(columns={'uv_by_country': 'Reach'})
        
    else:
        print('sainsbury is skipped ')
        # instead of pivot we can use the summed df above that already contains the sum over 
        # YT Service Code so the channels are already summarised in Services
        yt_deduped = summed_uv_by_country.rename(columns={'uv_by_country': 'Reach'})
    
    # processing 
    weekly_df= summary_excel(yt_deduped, bu, platformID, gam_info)
    
    # storing data
    data[bu]['weekly'] = weekly_df
    
    
    

### processing WSL ######################################################
{'Service IDs': ['AFA', 'FRE', 'AMH', 'ARA', 'AZE', 'BEN', 'POR', 'BUR', 'MAN', 'DAR', 'FAR', 'KRW', 'GUJ', 'HAU', 'HIN', 'IGB', 'INO', 'KOR', 'KYR', 'ELT', 'MAR', 'SPA', 'NEP', 'PAS', 'PER', 'PDG', 'POL', 'PUN', 'RUS', 'SER', 'SIN', 'SOM', 'SWA', 'TAM', 'TEL', 'THA', 'TIG', 'TUR', 'ECH', 'UKR', 'URD', 'UZB', 'VIE', 'YOR'], 'exclude_UK': False, 'sainsbury': {'TWI': True, 'YT-': True, 'FBE': True, 'TTK': True, 'TEL': True, 'INS': True}}
sainsbury is applied
saved weekly file for WSL as:
 ../data/singlePlatform/FBE/weekly//GAM2026_WEEKLY_FBE_WSLbyCountry.xlsx
### processing GNL_ ######################################################
{'Service IDs': ['BNI', 'BNO'], 'exclude_UK': True, 'sainsbury': {'TWI': True, 'YT-': True, 'FBE': True, 'TTK': True, 'TEL': True, 'INS': True}}
sainsbury is applied
saved weekly file for GNL_ as:
 ../data/singlePlatform/FBE/weekly//GAM2026_WEEKLY_FBE_GNL_byCountry.xlsx
### processing W

## AXE, GNL

In [10]:

data['AXE'] = {'weekly': pd.DataFrame()}
data['AXE']['weekly'] = calculate_weekly_Services(data['WSL']['weekly'], 'AXE', 
                                                            platformID, pop_size_col,
                                                            gam_info, 'sum') 

# TODO add explainer to make clear how a new service is created (BNI BNO breakout in GNL)
data['GNL'] = {'weekly': pd.DataFrame()}
data['GNL']['weekly'] = calculate_weekly_Services(data['GNL_']['weekly'], 'GNL', 
                                                            platformID, pop_size_col,
                                                            gam_info) 

## AX2, ANW, ANY, TOT, ALL, ENG, EN2 ENW


In [11]:
path = f"../data/singlePlatform/{platformID}/"
stages = [
        # (grouped_service, service1, service2, overlap_type, overlap_service_id, use_v2, optional_service3)
        ('AX2', 'FOA', 'AXE', 'WSL/FOA', 'FOA', False, None),
        ('ANW', 'AX2', 'WSE', 'WSE/WSL', 'AXE', False, None),
        ('ANY', 'GNL', 'ANW', 'WSL/GNL', 'ANW', False, None),
        ('TOT', 'MA-', 'ANY', 'sainsbury', '-', False, None),
        ('ALL', 'TOT', 'WOR', 'sainsbury', '-', False, None),
        ('ENG', 'GNL', 'WSE', 'sainsbury', '-', False, None),
        ('EN2', 'GNL', 'WSE', 'other', '-', True, 'WOR'),
        ('ENW', 'WSE', 'FOA', 'sainsbury', '-', False, None)
    ]
data = calculate_aggregated_services(data, stages, platformID, gam_info, path, overlaps, 
                                                country_codes, pop_size_col)


overlap applied: 0.035686
calculating annual by the new method
overlap applied: 0.071933


/Users/brunsd01/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/functions.py:496: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  year_avg = df.groupby(['ServiceID', 'PlaceID']).apply(compute_reach).reset_index(name='Reach')


calculating annual by the new method
overlap applied: 0.106934


/Users/brunsd01/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/functions.py:496: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  year_avg = df.groupby(['ServiceID', 'PlaceID']).apply(compute_reach).reset_index(name='Reach')


calculating annual by the new method
adding population: _merge
both          6084
left_only        0
right_only       0
Name: count, dtype: int64


...

/Users/brunsd01/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/functions.py:496: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  year_avg = df.groupby(['ServiceID', 'PlaceID']).apply(compute_reach).reset_index(name='Reach')


calculating annual by the new method
adding population: _merge
both          2601
left_only        0
right_only       0
Name: count, dtype: int64


/Users/brunsd01/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/functions.py:496: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  year_avg = df.groupby(['ServiceID', 'PlaceID']).apply(compute_reach).reset_index(name='Reach')


calculating annual by the new method


/Users/brunsd01/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/functions.py:496: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  year_avg = df.groupby(['ServiceID', 'PlaceID']).apply(compute_reach).reset_index(name='Reach')


In [12]:

# Combine all 'weekly' DataFrames
cols = ['PlaceID', 'ServiceID', 'w/c', 'Reach', ]
combined_weekly = pd.concat(
    [v['weekly'] for v in data.values() if 'weekly' in v],
    ignore_index=True
)[cols]


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['ServiceID'],
                                               main_data=combined_weekly,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['ServiceID', 'Start', 'End']],
                                               test_number=f"{platformID}_4_09",
                                               test_step="Check all weeks present for each account")


print(combined_weekly.shape)
combined_weekly.sample()
combined_weekly['PlatformID'] = platformID
# SERVICE hierarchy issues
test_step = "calculated high-level services"
service_hierarchy_issues = test_functions.test_hierarchy_reach(f"{platformID}_4_10", 
                                                               'Service', 
                                                               gam_info, 
                                                               combined_weekly, 
                                                               ['w/c', 'PlaceID'],
                                                               metric_col='Reach',
                                                               test_step= test_step, 
                                                                round_metric=True)


❌ Missing weeks from Start onward:
          Start End ServiceID        w/c
1701 2020-04-01 NaT       AFA 2025-12-15
700  2020-04-01 NaT       KRW 2025-12-15
310  2020-04-01 NaT       KYR 2025-12-15
271  2020-04-01 NaT       MA- 2025-12-15
1323 2020-04-01 NaT       MAN 2025-12-15
...         ...  ..       ...        ...
1636 2020-04-01 NaT       SER 2025-12-22
584  2020-04-01 NaT       IGB 2025-12-22
194  2020-04-01 NaT       SIN 2025-12-22
857  2020-04-01 NaT       PAS 2025-12-22
545  2020-04-01 NaT       YOR 2025-12-22

[94 rows x 4 columns]

Summary of missing weeks per channel_id_col:
   ServiceID  missing_week_count
0        AFA                   2
35       TAM                   2
26       PER                   2
27       POL                   2
28       PUN                   2
29       RUS                   2
30       SER                   2
31       SIN                   2
32       SOM                   2
33       SPA                   2
34       SWA                   2
36      

/Users/brunsd01/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/test_functions.py:115: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  main_test_data[key] = pd.to_datetime(main_test_data[key], errors='coerce', dayfirst=True)


...updating logbook...

✅ All tests passed.
